# Verifica con Nagini del numero di porte: algoritmo di Grover

 ## Notazione
 - n: numero di qubit del circuito (qubit di stato)
 - m: numero di qubit su cui agisce la riflessione, cioè `len(qubits)`
 - |O|: numero di porte dell'oracolo
 - k: numero di iterazioni di Grover  

## Premessa


- Cominciamo ad analizzare il codice di `GroverOperator`, che costruisce una **singola iterazione di Grover**
- Una singola iterazione applica in ordine:
    - l'oracolo, che marca le soluzioni con un segno -1
    - la trasformata di Hadamard $H^{⊗n}$
    - uno spostamento di fase condizionale, che lascia invariato $|0…0⟩$ e cambia segno a tutti gli altri stati
    - un'ulteriore trasformata di Hadamard $H^{⊗n}$

- Qiskit scrive l'iterazione come $Q = A S_0 A† S_f$, dove:
    -  $S_f$ è l'oracolo, 
    - $A = H^{⊗n}$ è la preparazione di stato 
    - $S_0$ è lo spostamento di fase condizionale (riflessione attorno a $|0…0⟩$)
- Gli ultimi tre passi insieme, $A S_0 A†$, formano l'inversione attorno alla media $2|ψ⟩⟨ψ| - I$


### Struttura di `GroverOperator`

(file `grover_originale/grover_operator_originale.py`)

- `__init__`: 
    - salva i parametri (oracolo, preparazione di stato, riflessione, qubit di riflessione, `insert_barriers`, `mcx_mode`) e chiama `_build`. 
    - Se l'oracolo o la riflessione sono dati come `Statevector` o `Operator`, li converte in un circuito `Diagonal`

- `reflection_qubits`: 
    - i qubit su cui agisce la riflessione $S_0$ di default tutti i qubit di stato, cioè `[0, ..., n-1]`
- `zero_reflection`: 
    - restituisce $S_0$. 
    - Se l'utente non l'ha fornita, la costruisce chiamando la funzione `_zero_reflection`
- `state_preparation`: 
    - restituisce A. 
    - Se l'utente non l'ha fornita, costruisce un circuito con H sui qubit di riflessione
- `oracle`: 
    - restituisce l'oracolo passato dall'utente, che marca con -1 le soluzioni. 
- `_build`: 
    - compone l'iterazione completa nell'ordine oracolo, $A†$ (`state_preparation.inverse()`), $S_0, A$
    - imposta la fase globale a $π$ per compensare il segno di $S_0$ e infine impacchetta tutto in un unico gate con `to_gate()`

**Ordine della verifica**
- Si parte dalle funzioni che non dipendono da altre: 
    - `_zero_reflection` 
    - la preparazione di stato
- poi quelle che le usano:
    - Poi `_build`, che le usa tramite le loro postcondizioni
    - Infine `construct_circuit` della classe `Grover`, che ripete k volte l'iterazione

# 1. `_zero_reflection`

- Parto da questa funzione perchè non ne richiama altre di Grover
- Costruisce $S_0$, la riflessione attorno a $|0…0⟩$ sui qubit indicati. 
    - l'operazione da svolgere è $2|0⟩⟨0| - I$ 
    - viene implementato $I - 2|0⟩⟨0|$ perchè è più semplice e perchè differisce di una fase globale (uno stato quantistico è definito a meno di una fase globale) 


## 1.1 Codice originale

File `grover_originale/grover_operator_originale.py`

In [ ]:
def _zero_reflection(
    num_state_qubits: int, qubits: List[int], mcx_mode: Optional[str] = None
) -> QuantumCircuit:
    qr_state = QuantumRegister(num_state_qubits, "state")
    reflection = QuantumCircuit(qr_state, name="S_0")

    num_ancillas = MCXGate.get_num_ancilla_qubits(len(qubits) - 1, mcx_mode)
    if num_ancillas > 0:
        qr_ancilla = AncillaRegister(num_ancillas, "ancilla")
        reflection.add_register(qr_ancilla)
    else:
        qr_ancilla = AncillaRegister(0)

    reflection.x(qubits)
    if len(qubits) == 1:
        reflection.z(0)  # MCX does not allow 0 control qubits, therefore this is separate
    else:
        reflection.h(qubits[-1])
        reflection.mcx(qubits[:-1], qubits[-1], qr_ancilla[:], mode=mcx_mode)
        reflection.h(qubits[-1])
    reflection.x(qubits)

    return reflection

## 1.2 Limite da dimostrare  

3 porte se m = 1 e 2m + 3 porte se m ≥ 2

- il numero di porte dipende solo da m, non dal numero totale di qubit del circuito

- In entrambi i casi il circuito è racchiuso tra due strati di X (bit flip) su tutti i qubit di riflessione: 2m porte
    - servono a scambiare |0…0⟩ e |1…1⟩, così il cambio di segno finisce su |0…0⟩
 
- In mezzo serve una porta che cambi segno solo a |1…1⟩ (phase flip)
- **Caso m = 1**: basta una Z sul qubit
    - totale: 2 X + 1 Z = 2m + 1 = 3 porte
- **Caso m ≥ 2**: serve una Z multi-controllata (controlli: tutti i qubit tranne l'ultimo; target: l'ultimo)
    - Qiskit la costruisce come H, MCX, H sul target, grazie all'uguaglianza HXH = Z
    - totale: 2m X + H + MCX + H = 2m + 3 porte

## 1.3 Metodi aggiunti allo stub
- il punto precedente fa vedere che vanno aggiunti dei metodi allo stub che era stato fatto per Deutsch. 
- conteneva x, h, cx e measure

**METODO Z** identico a h

In [ ]:
def z(self, qubit: int) -> None:
    Requires(Acc(self.num_qubits, 1/2) and Acc(self.gates))
    Requires(0 <= qubit and qubit < self.num_qubits)

    Ensures(Acc(self.num_qubits, 1/2) and Acc(self.gates))
    Ensures(self.gates == Old(self.gates) + 1)

    self.gates = self.gates + 1

**METODO X_LIST**
- Corrisponde a `x(lista)` di Qiskit, che aggiunge una porta X per ogni qubit della lista
- È un metodo separato da `x` per evitare problemi di tipo con Nagini (un parametro che può essere sia intero sia lista)
- Nella prima versione `x_list` chiedeva 1/2 del permesso, e Nagini segnalava che la precondizione di `x_list` poteva non valere alla seconda chiamata -> bisogna chiedere 1/4 del permesso

    -   **Causa**
        - `_zero_reflection` ha 1/2 del permesso su `qubits`; nella prima versione `x_list` chiedeva proprio 1/2, quindi la funzione cedeva tutto il suo permesso
        - Chi cede tutto il permesso, al ritorno sa della lista solo ciò che dice la postcondizione del metodo chiamato
        - La postcondizione di `x_list` non dice che la lista è rimasta uguale: Nagini ne dimentica contenuto e lunghezza, e alla seconda chiamata non riesce più a dimostrare che gli indici sono validi

    - **Correzione**
        - `x_list` (e `mcx`) chiedono 1/4; `_zero_reflection` resta a 1/2
        - Così la funzione cede 1/4 e tiene l'altro 1/4: la lista non può essere modificata e Nagini continua a conoscerne il contenuto

In [ ]:
def x_list(self, qubits: List[int]) -> None:
        Requires(Acc(self.num_qubits, 1/4) and Acc(self.gates))
        Requires(Acc(list_pred(qubits), 1/4))
        Requires(Forall(int, lambda i: Implies(0 <= i and i < len(qubits), 0 <= qubits[i] and qubits[i] < self.num_qubits)))

        Ensures(Acc(self.num_qubits, 1/4) and Acc(self.gates))
        Ensures(Acc(list_pred(qubits), 1/4))
        Ensures(self.gates == Old(self.gates) + len(qubits))

        self.gates = self.gates + len(qubits)

**METODO MCX:**
- Una sola porta, qualunque sia il numero di controlli
- `len(controls) >= 1`: MCX richiede almeno un controllo 
    - è il motivo per cui l'originale tratta a parte il caso m = 1
- Non richiede che il target sia diverso dai controlli

In [ ]:
def mcx(self, controls: List[int], target: int) -> None:
        Requires(Acc(self.num_qubits, 1/4) and Acc(self.gates))
        Requires(Acc(list_pred(controls), 1/4))
        Requires(len(controls) >= 1)
        Requires(Forall(int, lambda i: Implies(0 <= i and i < len(controls), 0 <= controls[i] and controls[i] < self.num_qubits)))
        Requires(0 <= target and target < self.num_qubits)

        Ensures(Acc(self.num_qubits, 1/4) and Acc(self.gates))
        Ensures(Acc(list_pred(controls), 1/4))
        Ensures(self.gates == Old(self.gates) + 1)

        self.gates = self.gates + 1

## 1.4 Trasformazioni


- **`mcx_mode` fissato a `"noancilla"`**: è il default di `GroverOperator`
    - con questa modalità le ancille sono sempre 0, quindi il parametro e tutto il blocco delle ancille spariscono
- **`QuantumCircuit(QuantumRegister(n, "state"), name="S_0")` → `QuantumCircuit(n, 0)`**: il modello non ha registri né nomi
- **`x(qubits)` → `x_list(qubits)`**: un metodo per ogni tipo di argomento
- **`qubits[-1]` e `qubits[:-1]` → `qubits[len(qubits) - 1]` e `qubits[:len(qubits) - 1]`**: indici espliciti, salvati in `target` e `controls`. Riscrittura preventiva, la forma originale non è stata provata con Nagini
- **`mcx(..., qr_ancilla[:], mode=mcx_mode)` → `mcx(controls, target)`**: il modello non ha ancille né modalità

**MCX conta come una porta**
- Sia il modello sia `size()` di Qiskit la contano come un'unica istruzione
- Contando la sua scomposizione in porte elementari, il limite dipenderebbe da m e dalla modalità scelta: è una decisione aperta

## 1.5 Versione verificata

File `groverVerificato.py`.

In [ ]:
def _zero_reflection(num_state_qubits: int, qubits: List[int]) -> QuantumCircuit:
    Requires(num_state_qubits > 0)
    Requires(Acc(list_pred(qubits), 1/2))
    Requires(len(qubits) >= 1)
    Requires(Forall(int, lambda i: Implies(0 <= i and i < len(qubits), 0 <= qubits[i] and qubits[i] < num_state_qubits)))

    Ensures(Acc(list_pred(qubits), 1/2))
    Ensures(Acc(Result().num_qubits) and Acc(Result().num_clbits) and Acc(Result().gates))
    Ensures(Result().num_qubits == num_state_qubits)
    Ensures(Implies(len(qubits) == 1, Result().gates == 3))
    Ensures(Implies(len(qubits) > 1, Result().gates == 2 * len(qubits) + 3))
    Ensures(Result().num_clbits == 0)

    reflection = QuantumCircuit(num_state_qubits, 0)

    reflection.x_list(qubits)
    if len(qubits) == 1:
        reflection.z(0)
    else:
        target = qubits[len(qubits) - 1]
        controls = qubits[:len(qubits) - 1]
        reflection.h(target)
        reflection.mcx(controls, target)
        reflection.h(target)
    reflection.x_list(qubits)

    return reflection

# 2. `_state_preparation`
- Costruisce A, la preparazione di stato: una porta H su ciascun qubit di riflessione
    - corrisponde alle due trasformate di Hadamard dell'iterazione di Grover, prima e dopo $S_0$
    - in `_build` viene usata due volte: invertita (A†) prima di $S_0$ e normale (A) dopo
    - siccome $H† = H$, qui A† e A contengono le stesse porte
- Come `_zero_reflection`, non chiama altre funzioni di Grover, quindi si può verificare direttamente


## 2.1 Codice originale
File `grover_originale/grover_operator_originale.py`

In [ ]:
@property
def state_preparation(self) -> QuantumCircuit:
    """The subcircuit implementing the A operator or Hadamards."""
    if self._state_preparation is not None:
        return self._state_preparation

    num_state_qubits = self.oracle.num_qubits - self.oracle.num_ancillas
    hadamards = QuantumCircuit(num_state_qubits, name="H")
    # apply Hadamards only on reflection qubits, rest will cancel out
    hadamards.h(self.reflection_qubits)
    return hadamards


- Se l'utente ha fornito una sua preparazione di stato, la restituisce così com'è
- Altrimenti calcola il numero di qubit di stato dall'oracolo e crea un circuito con H sui qubit di riflessione
- H va solo sui qubit di riflessione, perchè sugli altri qubit $S_0$ non agisce
    - quindi le $H$ di $A†$ e di $A$ si annullerebbero ($H H = I$)

## 2.2 Limite da dimostrare

m porte, con $m = len(qubits)$

- Una H per ogni qubit di riflessione
- Il numero di porte dipende solo da m, non dal numero totale di qubit del circuito
- Non ci sono casi separati: la formula vale per ogni m ≥ 1

## 2.3 Metodi aggiunti allo stub

**METODO H_LIST**

- Corrisponde a `h(lista)` di Qiskit: aggiunge una porta H per ogni qubit della lista
- È identico a `x_list`, cambia solo il nome
- Chiede 1/4 del permesso sulla lista per lo stesso motivo di `x_list`

In [ ]:
def h_list(self, qubits: List[int]) -> None:
    Requires(Acc(self.num_qubits, 1/4) and Acc(self.gates))
    Requires(Acc(list_pred(qubits), 1/4))
    Requires(Forall(int, lambda i: Implies(0 <= i and i < len(qubits), 0 <= qubits[i] and qubits[i] < self.num_qubits)))

    Ensures(Acc(self.num_qubits, 1/4) and Acc(self.gates))
    Ensures(Acc(list_pred(qubits), 1/4))
    Ensures(self.gates == Old(self.gates) + len(qubits))

    self.gates = self.gates + len(qubits)

## 2.4 Trasformazioni


- **Property → funzione libera `_state_preparation(num_state_qubits, qubits)`**: la property legge tutto da `self`, ma il numero di porte dipende solo dal numero di qubit e dalla lista dei qubit. Passandoli come parametri non serve modellare l'intera classe `GroverOperator`
    - `self.oracle.num_qubits - self.oracle.num_ancillas` → `num_state_qubits` (senza ancille è il numero di qubit dell'oracolo)
    - `self.reflection_qubits` → `qubits`
    - sarà `_build` a passare questi valori
- **Preparazione fornita dall'utente rimossa**: avrebbe un numero di porte non noto; si considera il Grover standard con le H
- **`QuantumCircuit(num_state_qubits, name="H")` → `QuantumCircuit(num_state_qubits, 0)`**: il modello non ha nomi
- **`h(lista)` → `h_list(lista)`**: un metodo per ogni tipo di argomento

## 2.5 Versione verificata
File `groverVerificato.py`.

In [ ]:
def _state_preparation(num_state_qubits: int, qubits: List[int]) -> QuantumCircuit:
    Requires(num_state_qubits > 0)
    Requires(Acc(list_pred(qubits), 1/2))
    Requires(len(qubits) >= 1)
    Requires(Forall(int, lambda i: Implies(0 <= i and i < len(qubits),0 <= qubits[i] and qubits[i] < num_state_qubits)))

    Ensures(Acc(list_pred(qubits), 1/2))
    Ensures(Acc(Result().num_qubits) and Acc(Result().num_clbits) and Acc(Result().gates))
    Ensures(Result().num_qubits == num_state_qubits)
    Ensures(Result().gates == len(qubits))
    Ensures(Result().num_clbits == 0)

    hadamards = QuantumCircuit(num_state_qubits, 0)
    hadamards.h_list(qubits)
    return hadamards

- Le precondizioni sono le stesse di `_zero_reflection`, perché le due funzioni ricevono gli stessi dati
- Il costruttore parte da 0 porte e `h_list` ne aggiunge len(qubits): da qui `Result().gates == len(qubits)`
- `Result().num_clbits == 0` è stata aggiunta durante l'analisi di `_build`-> il risultato di `_state_preparation` viene invertito con `inverse()`, che richiede `num_clbits >= 0`

# 3. `_build`

- Costruisce l'intera iterazione di Grover: $Q = A S_0 A† S_f$
- Mette insieme l'oracolo e le due funzioni già verificate (`_state_preparation` e `_zero_reflection`)
- Nagini non rientra nel corpo di quelle funzioni -> usa solo le loro postcondizioni. 
    - Per questo andavano verificate prima
- È il primo livello in cui si combinano più circuiti, quindi servono `inverse` e `compose`


## 3.1 Codice originale

- File `grover_originale/grover_operator_originale.py`
- È un metodo della classe `GroverOperator`, chiamato da `__init__`.


In [ ]:
def _build(self):
    num_state_qubits = self.oracle.num_qubits - self.oracle.num_ancillas
    circuit = QuantumCircuit(QuantumRegister(num_state_qubits, name="state"), name="Q")
    num_ancillas = numpy.max(
        [
            self.oracle.num_ancillas,
            self.zero_reflection.num_ancillas,
            self.state_preparation.num_ancillas,
        ]
    )
    if num_ancillas > 0:
        circuit.add_register(AncillaRegister(num_ancillas, name="ancilla"))

    circuit.compose(self.oracle, list(range(self.oracle.num_qubits)), inplace=True)
    if self._insert_barriers:
        circuit.barrier()
    circuit.compose(
        self.state_preparation.inverse(),
        list(range(self.state_preparation.num_qubits)),
        inplace=True,
    )
    if self._insert_barriers:
        circuit.barrier()
    circuit.compose(
        self.zero_reflection, list(range(self.zero_reflection.num_qubits)), inplace=True
    )
    if self._insert_barriers:
        circuit.barrier()
    circuit.compose(
        self.state_preparation, list(range(self.state_preparation.num_qubits)), inplace=True
    )

    # minus sign
    circuit.global_phase = numpy.pi

    self.add_register(*circuit.qregs)
    try:
        circuit_wrapped = circuit.to_gate()
    except QiskitError:
        circuit_wrapped = circuit.to_instruction()

    self.compose(circuit_wrapped, qubits=self.qubits, inplace=True)

- Crea un circuito con i qubit di stato e, se servono, aggiunge le ancille 
    - il massimo tra quelle di oracolo, $S_0$ e $A$
- Compone in ordine: oracolo, $A†$ (`state_preparation.inverse()`), $S_0, A$
    - se `insert_barriers` è vero, mette una barrier tra un blocco e l'altro
- Imposta la fase globale a π, per compensare il segno di $S_0$ (Qiskit implementa $I - 2|0⟩⟨0|$ invece di $2|0⟩⟨0| - I$) -> spiegato in `_zero_reflection`
- Impacchetta tutto il circuito in un unico gate con `to_gate()` e lo aggiunge a `self`
    - se prima avevamo diverse porte ora vengono tutte compattate e ci si riferisce a loro come un'unica istruzione di GroverOperator
        - se facciamo `GroverOperator(oracle).size()` dà 1.
        - Per vedere le porte effettivamente usate bisogna aprire la scatola con `decompose()`: `GroverOperator(oracle).decompose().size()` dà $|O| + 4m + 3$.



## 3.2 Limite da dimostrare


- $|O| + 5$ porte se $m = 1$ 
- e $|O| + 4m + 3$ porte se $m ≥ 2$

* Il conteggio si ottiene sommando i quattro blocchi (la dimensione dei vari blocchi è spiegata nelle 2 sezioni precedenti)
    - Oracolo $S_f$: |O| porte
    - $A†$ (`_state_preparation` invertita): m porte (1 se m = 1)
    - $S_0$ (`_zero_reflection`): 2m + 3 porte (3 se m = 1)
    - $A$ (`_state_preparation`): m porte (1 se m = 1)
    - **Totale**: $|O| + 4m + 3$ se m ≥ 2, $|O| + 5$ se m = 1




## 3.3 Metodi aggiunti allo stub

**METODO INVERSE**
- Restituisce un circuito nuovo con gli stessi qubit, gli stessi bit classici e lo stesso numero di porte: l'inversa sostituisce ogni porta con la sua inversa e ribalta l'ordine
- Su `self` chiede solo 1/4 dei permessi, perché lo legge senza modificarlo
- Sul risultato dà il permesso pieno: è un circuito nuovo, che chi chiama può modificare. 
    - Nagini deduce da solo che è diverso da `self`, perché non si può avere permesso pieno su un oggetto e contemporaneamente 1/4 sullo stesso
- `num_qubits > 0` e `num_clbits >= 0` servono al costruttore usato nel corpo


In [ ]:
def inverse(self) -> 'QuantumCircuit':
    Requires(Acc(self.num_qubits, 1/4) and Acc(self.num_clbits, 1/4) and Acc(self.gates, 1/4))
    Requires(self.num_qubits > 0 and self.num_clbits >= 0)

    Ensures(Acc(self.num_qubits, 1/4) and Acc(self.num_clbits, 1/4) and Acc(self.gates, 1/4))
    Ensures(Acc(Result().num_qubits) and Acc(Result().num_clbits) and Acc(Result().gates))
    Ensures(Result().num_qubits == self.num_qubits)
    Ensures(Result().num_clbits == self.num_clbits)
    Ensures(Result().gates == self.gates)

    inv = QuantumCircuit(self.num_qubits, self.num_clbits)
    inv.gates = self.gates
    return inv

**METODO COMPOSE**
- Corrisponde a `compose(other, qubits, inplace=True)` di Qiskit: aggiunge a `self` le porte di `other`
- Le porte di `self` diventano le vecchie più quelle di `other`
- Permesso pieno su `self.gates`, che viene modificato; 1/4 su `other`, che viene solo letto
- `other` deve avere al massimo tanti qubit quanti `self`
- Effetto dei permessi: non si può comporre un circuito con se stesso (servirebbe più del permesso totale su `gates`); Grover non lo fa mai


In [ ]:
def compose(self, other: 'QuantumCircuit') -> None:
    Requires(Acc(self.num_qubits, 1/4) and Acc(self.gates))
    Requires(Acc(other.num_qubits, 1/4) and Acc(other.gates, 1/4))
    Requires(other.num_qubits <= self.num_qubits)

    Ensures(Acc(self.num_qubits, 1/4) and Acc(self.gates))
    Ensures(Acc(other.num_qubits, 1/4) and Acc(other.gates, 1/4))
    Ensures(self.gates == Old(self.gates) + other.gates)

    self.gates = self.gates + other.gates

**Semplificazioni del modello**
- `compose` ignora la lista dei qubit su cui va il circuito composto e i bit classici; modella solo `inplace=True`
- `inverse` non modella le singole porte invertite, solo il loro numero


## 3.4 Trasformazioni

- **Metodo → funzione libera `_build(oracle, qubits)`**: come per `_state_preparation`, i dati letti da `self` diventano parametri
    - `self.oracle.num_qubits - self.oracle.num_ancillas` → `n = oracle.num_qubits` (il modello non ha ancille)
    - le property `self.state_preparation` (usata due volte) e `self.zero_reflection` → chiamate a `_state_preparation(n, qubits)` (due volte) e `_zero_reflection(n, qubits)`, già verificate
- **`QuantumCircuit(QuantumRegister(n, name="state"), name="Q")` → `QuantumCircuit(n, 0)`**: il modello non ha registri né nomi
- **Ancille rimosse** (`numpy.max(...)` e `add_register(...)`): con `"noancilla"` e un oracolo senza ancille sono 0
- **Barrier rimosse**: il default è `insert_barriers = False`; se contarle è una decisione aperta
- **`compose(x, list(range(...)), inplace=True)` → `compose(x)`**: il modello non registra su quali qubit va il circuito composto
- **`global_phase = numpy.pi` rimossa**: è una fase globale, non aggiunge porte
- **Impacchettamento rimosso** (`to_gate()` e `self.compose(circuit_wrapped, ...)`): la funzione restituisce direttamente `circuit`, così si conta il numero reale di porte e non un'unica istruzione

## 3.5 Versione verificata

File `groverVerificato.py`.

In [ ]:
def _build(oracle: QuantumCircuit, qubits: List[int]) -> QuantumCircuit:
    Requires(Acc(oracle.num_qubits, 1/2) and Acc(oracle.gates, 1/2))
    Requires(oracle.num_qubits > 0)
    Requires(Acc(list_pred(qubits)))
    Requires(len(qubits) >= 1)
    Requires(Forall(int, lambda i: Implies(0 <= i and i < len(qubits), 0 <= qubits[i] and qubits[i] < oracle.num_qubits)))

    Ensures(Acc(oracle.num_qubits, 1/2) and Acc(oracle.gates, 1/2))
    Ensures(Acc(list_pred(qubits)))
    Ensures(Acc(Result().num_qubits) and Acc(Result().num_clbits) and Acc(Result().gates))
    Ensures(Result().num_qubits == oracle.num_qubits)
    Ensures(Implies(len(qubits) == 1, Result().gates == oracle.gates + 5))
    Ensures(Implies(len(qubits) > 1, Result().gates == oracle.gates + 4 * len(qubits) + 3))
    Ensures(Result().num_clbits == 0)
    Ensures(len(qubits) == Old(len(qubits)))

    n = oracle.num_qubits
    circuit = QuantumCircuit(n, 0)

    circuit.compose(oracle)

    a = _state_preparation(n, qubits)
    a_inv = a.inverse()
    circuit.compose(a_inv)

    s0 = _zero_reflection(n, qubits)
    circuit.compose(s0)

    a2 = _state_preparation(n, qubits)
    circuit.compose(a2)

    return circuit

- **Permessi**: chi sta più in alto nella catena di chiamate ha più permesso sulla lista (`_build` 1, `_zero_reflection` e `_state_preparation` 1/2, metodi dello stub 1/4)
    - oracolo a 1/2: viene solo letto
    - `qubits` pieno: `_build` passa la lista a tre funzioni che ne chiedono 1/2 ciascuna; con il permesso pieno ne tiene sempre metà e non perde le informazioni sulla lista 
- **`Forall`**: gli indici sono validi rispetto a `oracle.num_qubits`, cioè il numero di qubit di tutti i circuiti coinvolti
- **Limite**: `oracle.gates` è |O|; i due casi sono separati con `Implies`
- **Postcondizioni aggiunte durante `construct_circuit`**:
    - `Result().num_clbits == 0`: richiesta da `power`
    - `len(qubits) == Old(len(qubits))`: chi chiama `_build` le cede tutta la lista, quindi ha bisogno che sia `_build` a garantire che la lunghezza non è cambiata

# 4. `construct_circuit`

- Costruisce il circuito completo di Grover -> la preparazione di stato + k iterazioni
- Usa `_state_preparation` e `_build`, già verificate, più due elementi nuovi:
    1. il metodo `power` nello stub, che ripete un circuito k volte
    2. la funzione ausiliaria `_all_qubits`, che costruisce la lista [0, ..., n-1]
* `power` e `_all_qubits` contengono i primi invarianti di ciclo di Grover
* Adesso il limite riguarda l'intero circuito che la classe `Grover` costruisce, non più una singola iterazione


## 4.1 Codice originale

File `grover_originale/grover_originale.py`


In [ ]:
def construct_circuit(
    self, problem: AmplificationProblem, power: int | None = None, measurement: bool = False
) -> QuantumCircuit:
    if power is None:
        if len(self._iterations) > 1:
            raise ValueError("Please pass ``power`` if the iterations are not an integer.")
        power = self._iterations[0]

    qc = QuantumCircuit(problem.oracle.num_qubits, name="Grover circuit")
    qc.compose(problem.state_preparation, inplace=True)
    if power > 0:
        qc.compose(problem.grover_operator.power(power), inplace=True)

    if measurement:
        measurement_cr = ClassicalRegister(len(problem.objective_qubits))
        qc.add_register(measurement_cr)
        qc.measure(problem.objective_qubits, measurement_cr)

    return qc

- Se il numero di iterazioni `power` non è dato, lo prende da quelle configurate nella classe
- Crea un circuito con tanti qubit quanti l'oracolo
- Compone la preparazione di stato
- Se `power > 0`, compone l'operatore di Grover ripetuto `power` volte
- Se `measurement` è vero, aggiunge un registro classico e misura i qubit obiettivo

**DOV'È IL CICLO?**
- Non c'è un ciclo esplicito: le k iterazioni stanno in `grover_operator.power(power)`
- In Qiskit `power` con un esponente intero chiama `repeat`, che impacchetta il circuito in un gate e lo aggiunge k volte

### 4.1.1 AmplificationProblem

**Perché serve `AmplificationProblem`**
- `construct_circuit` riceve un oggetto `problem` e ne usa preparazione di stato, operatore di Grover e qubit da misurare
- Se l'utente passa solo l'oracolo, questi vengono costruiti con valori di default: per contare le porte bisogna conoscerli

**Cosa ho conrollato** (`grover_originale/amplification_problem_originale.py`)
- I default delle tre property:
    - `state_preparation`: H su tutti gli n qubit
    - `grover_operator`: `GroverOperator(oracle, state_preparation)`, con la riflessione su tutti gli n qubit
    - `objective_qubits`: tutti gli n qubit
- Il resto della classe non costruisce circuiti

**Come lo uso**
- Nel caso di default m = n e `qubits = [0, ..., n-1]`, costruita con `_all_qubits(n)`
- `problem` viene sostituito dal solo oracolo:
    - preparazione → `_state_preparation(n, qubits)` 
    - operatore → `_build(oracle, qubits)`, con |O| + 4n + 3 porte per iterazione
- Esclusi: le misure (aggiungerebbero n porte) e le preparazioni o gli operatori forniti dall'utente

## 4.2 Limite da dimostrare

- $n + k(|O| + 4n + 3)$ porte se $n ≥ 2$ 
- $1 + k(|O| + 5)$ porte se $n = 1$
- con n = qubit dell'oracolo, |O| = porte dell'oracolo, k = numero di iterazioni


+ Preparazione di stato: n porte
+ k iterazioni di Grover, ognuna con il numero di porte del livello 3 
+ Con k = 0 resta solo la preparazione, e la formula dà n
+ Il limite dipende solo dai parametri d'ingresso: n, |O| e k

* **Misure non contate**: il limite riguarda il circuito con `measurement=False` (il default)
    * con `measurement=True` Qiskit aggiungerebbe una misura per ciascuno degli n qubit, cioè n istruzioni in più
* Come al livello 3, si contano le **porte srotolate**

## 4.3 Metodi aggiunti allo stub

**METODO POWER**
- Corrisponde a `power(k)` di Qiskit: restituisce un circuito nuovo che contiene k copie di `self`
- Nel modello è srotolato: le porte del risultato sono k volte quelle di `self` 
    - in Qiskit, con l'impacchettamento, sarebbero k istruzioni
- Il corpo è un ciclo che compone `self` k volte in un circuito inizialmente vuoto


In [ ]:
def power(self, k: int) -> 'QuantumCircuit':
    Requires(Acc(self.num_qubits, 1/2) and Acc(self.num_clbits, 1/2) and Acc(self.gates, 1/2))
    Requires(self.num_qubits > 0 and self.num_clbits >= 0)
    Requires(k >= 0)

    Ensures(Acc(self.num_qubits, 1/2) and Acc(self.num_clbits, 1/2) and Acc(self.gates, 1/2))
    Ensures(Acc(Result().num_qubits) and Acc(Result().num_clbits) and Acc(Result().gates))
    Ensures(Result().num_qubits == self.num_qubits)
    Ensures(Result().num_clbits == self.num_clbits)
    Ensures(Result().gates == k * self.gates)

    rep = QuantumCircuit(self.num_qubits, self.num_clbits)
    i = 0
    while i < k:
        Invariant(Acc(self.num_qubits, 1/2) and Acc(self.num_clbits, 1/2) and Acc(self.gates, 1/2))
        Invariant(Acc(rep.num_qubits) and Acc(rep.num_clbits) and Acc(rep.gates))
        Invariant(rep.num_qubits == self.num_qubits and rep.num_clbits == self.num_clbits)
        Invariant(0 <= i and i <= k)
        Invariant(rep.gates == i * self.gates)
        rep.compose(self)
        i = i + 1
    return rep

- **Permesso 1/2 su `self`**: a ogni giro `compose` ne prende 1/4, e `power` tiene l'altro 1/4, così `self.gates` non cambia tra un giro e l'altro 
- **Permessi nell'invariante**: Nagini verifica il corpo del ciclo a parte e vede solo ciò che dice l'invariante, quindi i permessi vanno ripetuti lì
- **`rep.gates == i * self.gates`**: dopo i giri `rep` contiene i copie di `self`
    - all'uscita i = k, quindi `rep.gates == k * self.gates`
- **`rep.num_qubits == self.num_qubits`**: serve alla precondizione di `compose`

## 4.4 Funzione ausiliaria `_all_qubits`


- Costruisce la lista [0, ..., n-1]
    - cioè i qubit di riflessione di default che Qiskit ottiene con `list(range(n))`
- È scritta con un ciclo esplicito: l'invariante dà direttamente le proprietà che servono a `_state_preparation` e `_build`, e chi la chiama ottiene il permesso pieno sulla lista


In [ ]:
def _all_qubits(n: int) -> List[int]:
    Requires(n > 0)
    Ensures(Acc(list_pred(Result())))
    Ensures(len(Result()) == n)
    Ensures(Forall(int, lambda j: Implies(0 <= j and j < n, Result()[j] == j)))

    qubits: List[int] = []
    i = 0
    while i < n:
        Invariant(Acc(list_pred(qubits)))
        Invariant(0 <= i and i <= n)
        Invariant(len(qubits) == i)
        Invariant(Forall(int, lambda j: Implies(0 <= j and j < i, qubits[j] == j)))
        qubits.append(i)
        i = i + 1
    return qubits

- Dopo i giri la lista ha lunghezza i e contiene esattamente 0, ..., i-1
- All'uscita vale i = n, quindi la lista è [0, ..., n-1]
- Da `Result()[j] == j` Nagini ricava il `Forall` "indici validi" richiesto da `_state_preparation` e `_build`: ogni elemento è compreso tra 0 e n-1


## 4.5 Trasformazioni

- **Metodo → funzione libera `construct_circuit(oracle, k)`**
    - L'originale riceve un oggetto `problem` (di tipo `AmplificationProblem`) e da lì prende la preparazione di stato e l'operatore di Grover
    - Se l'utente fornisce solo l'oracolo, questi due pezzi sono costruiti con i valori di default, che dipendono solo dall'oracolo
    - Quindi nella versione verificata basta passare l'oracolo, e i due pezzi si ricostruiscono con le funzioni già verificate:
        - `problem.state_preparation` → `_state_preparation(n, qubits)`
        - `problem.grover_operator` → `_build(oracle, qubits)` 
    - Le funzioni  verificate vogliono la lista dei qubit come parametro, mentre in Qiskit "tutti i qubit" è implicito nei default e non viene mai scritto. 
        - La lista [0, ..., n-1] va quindi costruita esplicitamente: `qubits = _all_qubits(n)`

- **Numero di iterazioni**: il ramo `if power is None` è rimosso, k è sempre passato come parametro
- **il modello non ha nomi**`QuantumCircuit(problem.oracle.num_qubits, name=...)` → `QuantumCircuit(n, 0)`
- **`grover_operator.power(power)` → `q.power(k)`**, srotolato come nel punto 3
- **Misure rimosse**: il default è `measurement=False`, e il modello non ha registri classici
- **`compose(x, inplace=True)` → `compose(x)`**: il modello prevede solo `inplace=True`

## 4.6 Versione verificata

File `groverVerificato.py`.

In [ ]:
def construct_circuit(oracle: QuantumCircuit, k: int) -> QuantumCircuit:
    Requires(Acc(oracle.num_qubits) and Acc(oracle.gates))
    Requires(oracle.num_qubits > 0)
    Requires(k >= 0)

    Ensures(Acc(oracle.num_qubits) and Acc(oracle.gates))
    Ensures(Acc(Result().num_qubits) and Acc(Result().num_clbits) and Acc(Result().gates))
    Ensures(Result().num_qubits == oracle.num_qubits)
    Ensures(Implies(oracle.num_qubits == 1, Result().gates == 1 + k * (oracle.gates + 5)))
    Ensures(Implies(oracle.num_qubits > 1, Result().gates == oracle.num_qubits + k * (oracle.gates + 4 * oracle.num_qubits + 3)))

    n = oracle.num_qubits
    qubits = _all_qubits(n)

    qc = QuantumCircuit(n, 0)
    prep = _state_preparation(n, qubits)
    qc.compose(prep)

    if k > 0:
        q = _build(oracle, qubits)
        qk = q.power(k)
        qc.compose(qk)

    return qc

**Contratti**
- **Oracolo, permesso pieno**: `construct_circuit` è in cima alla catena; `_build` ne prende metà e l'altra metà tiene noto `oracle.gates` fino alla postcondizione
- **`k >= 0`**: richiesto da `power`
- **Lista `qubits`**: creata dentro la funzione con permesso pieno
    - `_state_preparation` ne prende metà, quindi lunghezza e contenuto restano noti
    - `_build` la prende tutta: per questo garantisce `len(qubits) == Old(len(qubits))`, che permette di sostituire `len(qubits)` con n
- **`power`**: richiede `num_clbits >= 0`, garantito da `_build`
- **Limite**: espresso solo con i parametri d'ingresso n, |O| e k

**Note**
- In Qiskit, contando le porte impacchettate, il circuito ha n + k istruzioni
    - bisogna decidere se tenere le porte impacchettate o srotolate 
- Il risultato copre il circuito completo di Grover per ogni k ≥ 0, nel caso di default (H e riflessione su tutti i qubit, `"noancilla"`, nessuna misura)

# Riepilogo modifiche principali al codice originale

**Opzioni non di default rimosse**
- `mcx_mode` fissato a `"noancilla"`, quindi nessuna ancilla
- Preparazione di stato e operatore di Grover forniti dall'utente: si considerano solo quelli di default
- Barrier (`insert_barriers = False`) e misure (`measurement = False`)

**Cambi di struttura**
- Property e metodi che leggono da `self` diventano funzioni libere con parametri espliciti (`_state_preparation`, `_build`, `construct_circuit`)
- L'oggetto `problem` è sostituito dal solo oracolo più il numero di iterazioni k
- La lista dei qubit, implicita nei default di Qiskit, diventa esplicita con `_all_qubits(n)`

**Adattamenti al modello**
- Niente registri né nomi: `QuantumCircuit(n, 0)`
- Metodi separati per le liste: `x_list`, `h_list`
- Indici negativi e slicing riscritti con indici espliciti
- `compose` senza la lista dei qubit, solo `inplace=True`

**Rimossi perché non aggiungono porte**
- `global_phase`

**Impacchettamento rimosso**
- `to_gate()` in `_build` e `repeat` dentro `power`: si contano le porte srotolate

**Metodi aggiunti allo stub**
- `z`, `x_list`, `h_list`, `mcx`, `inverse`, `compose`, `power` (con invariante di ciclo)